In [0]:
read_path="s3://olist-batch-processing-project-bucket/bronze"
write_path="s3://olist-batch-processing-project-bucket/silver"

customers_read_path = f"{read_path}/customers_dataset.csv"
geolocation_read_path = f"{read_path}/geolocation_dataset.csv"
order_items_read_path = f"{read_path}/order_items_dataset.csv"
order_payments_read_path = f"{read_path}/order_payments_dataset.csv"
order_reviews_read_path = f"{read_path}/order_reviews_dataset.csv"
orders_read_path = f"{read_path}/orders_dataset.csv"
product_category_name_translation_read_path = f"{read_path}/product_category_name_translation.csv"
products_read_path = f"{read_path}/products_dataset.csv"
sellers_read_path = f"{read_path}/sellers_dataset.csv"


customers_write_path = f"{write_path}/customers_data"
geolocation_write_path = f"{write_path}/geolocation_data"
order_items_write_path = f"{write_path}/order_items_data"
order_payments_write_path = f"{write_path}/order_payments_data"
order_reviews_write_path = f"{write_path}/order_reviews_data"
orders_write_path = f"{write_path}/orders_data"
product_category_name_translation_write_path = f"{write_path}/product_category_name_translation_data"
products_write_path = f"{write_path}/products_data"
sellers_write_path = f"{write_path}/sellers_data"



In [0]:
import pyspark.sql.functions as sf

# CUSTOMER DATA TRANSFORMATION

In [0]:
df_customers= spark.read.csv(customers_read_path, header=True, inferSchema=True)

In [0]:
df_customers= df_customers.dropDuplicates(["customer_id"])

# customer unique id is unique to each user
# While customer id is used for the relation with the order table, there are duplicates in some unique ids because a new customer id is created for each order
# For a user who placed 2 orders, 2 separate customer ids were created but the same customer unique id was used, therefore we should not remove the duplications in customer id
# Therefore, while the unique id can have duplicates, the customer id cannot have duplicates

df_customers= df_customers.where(
    (sf.col("customer_id").isNotNull())&
    (sf.trim(sf.col("customer_id")) != "")&
    ~(sf.col("customer_id").contains(" "))& 
    (sf.col("customer_unique_id").isNotNull())&
    (sf.trim(sf.col("customer_unique_id")!=""))&
    ~(sf.col("customer_unique_id").contains(" "))& 
    (sf.col("customer_zip_code_prefix").isNotNull())& 
    ((sf.col("customer_zip_code_prefix")>=1001) & (sf.col("customer_zip_code_prefix")<=99990))
)

df_customers= df_customers.select(
                            sf.lower(sf.trim("customer_id")).alias("customer_id")
                            ,sf.lower(sf.trim(sf.col("customer_unique_id"))).alias("customer_unique_id")
                            ,"customer_zip_code_prefix" 
                            # "It is not important for the city and state to be null; we will get the location information via the zip code."
                            ,sf.when(sf.col("customer_city").isNull(),"UNKNOWN").otherwise(sf.lower(sf.trim(sf.col("customer_city")))).alias("customer_city")
                            ,sf.when(sf.col("customer_state").isNull(),"UNKNOWN").otherwise(sf.upper(sf.trim(sf.col("customer_state")))).alias("customer_state")
                            )

In [0]:
df_customers.write.mode("overwrite").parquet(customers_write_path)

# GEOLOCATION DATA TRANSFORMATION

In [0]:
#df_geolocation.orderBy(sf.col("geolocation_zip_code_prefix")).show(1)      ---- min zip code (1001)
#df_geolocation.orderBy(sf.col("geolocation_zip_code_prefix").desc()).show(1, truncate=False)   ----- max zip code (99990)

# MAX and MIN zip codes for validation

In [0]:
df_geolocation= spark.read.csv(geolocation_read_path, header=True, inferSchema=True)

In [0]:
df_geolocation=df_geolocation.distinct()

# "We have duplicate zip codes. We cannot pinpoint every location using the available zip code information. Therefore, instead of keeping all of them,
# keeping one generalized entry will be sufficient to know the location to a certain extent. We are getting rid of redundant data instead of keeping 
# duplicates. For this reason, let's take the entries with the same zip code, average them (AVG), and convert them into a single entry. Since we are 
# storing an approximate location

df_geolocation = df_geolocation.groupBy(
    "geolocation_zip_code_prefix"
).agg(
    sf.avg("geolocation_lat").alias("geolocation_lat"),
    sf.avg("geolocation_lng").alias("geolocation_lng"),
    # Using 'first' is a common way to select an arbitrary value in a group.
    sf.first(sf.lower(sf.trim("geolocation_city"))).alias("geolocation_city"),
    sf.first(sf.upper(sf.trim("geolocation_state"))).alias("geolocation_state")
)


df_geolocation= df_geolocation.where(
    ((sf.col("geolocation_lat").isNotNull()) &
    (sf.col("geolocation_lng").isNotNull()) &
     (sf.col("geolocation_zip_code_prefix").isNotNull()) & 
     (sf.col("geolocation_state").isNotNull()) & 
     (sf.col("geolocation_city").isNotNull()) &
     (sf.trim(sf.col("geolocation_city"))!="")&
     (sf.trim(sf.col("geolocation_state"))!="")
    )
)

df_geolocation= df_geolocation.select(
                                        "geolocation_zip_code_prefix"
                                        ,sf.round(sf.col("geolocation_lat"), 5).alias("geolocation_lat") 
                                        # we don't need to keep all the digits after the decimal point.
                                        ,sf.round(sf.col("geolocation_lng"), 5).alias("geolocation_lng")
                                        ,sf.lower(sf.trim("geolocation_city")).alias("geolocation_city")
                                        ,sf.upper(sf.trim("geolocation_state")).alias("geolocation_state")
                                      )



In [0]:
df_geolocation.write.mode("overwrite").parquet(geolocation_write_path)

# SELLERS TRANSFORMATION

In [0]:
df_sellers= spark.read.csv(sellers_read_path, header=True, inferSchema=True)

In [0]:
df_sellers=df_sellers.distinct()

df_sellers=df_sellers.dropDuplicates(["seller_id"])

df_sellers=df_sellers.where(
    (sf.col("seller_zip_code_prefix") <= 99990) & (sf.col("seller_zip_code_prefix") >= 1001)& # min and max zip code values are 1001 and 99990
    (sf.col("seller_zip_code_prefix").isNotNull())&
    (sf.col("seller_id").isNotNull()) &
    (sf.trim(sf.col("seller_id") !=""))&
    ~(sf.col("seller_id").contains(" "))
)

df_sellers=df_sellers.select(
    sf.lower(sf.trim(sf.col("seller_id")).alias("seller_id")).alias("seller_id"),
    "seller_zip_code_prefix",
    sf.when(sf.col("seller_city").isNull(),"UNKNOWN").otherwise(sf.lower(sf.trim(sf.col("seller_city")))).alias("seller_city"), 
    sf.when(sf.col("seller_state").isNull(),"UNKNOWN").otherwise(sf.upper(sf.trim(sf.col("seller_state")))).alias("seller_state") 
    # "It is not important for the city and state to be null; we will get the location information via the zip code."
)

In [0]:
df_sellers.write.mode("overwrite").parquet(sellers_write_path)

# PRODUCTS TRANSFORMATION

In [0]:
df_products= spark.read.csv(products_read_path, header=True, inferSchema=True)

In [0]:
df_products=df_products.distinct()

df_products=df_products.dropDuplicates(["product_id"])

df_products= df_products.where(
        (sf.col("product_id").isNotNull())&
        (sf.trim(sf.col("product_id") !=""))&
        ~(sf.col("product_id").contains(" "))
        )

df_products=df_products.select(
    sf.trim(sf.col("product_id")).alias("product_id"),

    sf.when(sf.col("product_category_name").isNull(),sf.lit("UNKNOWN"))
        .otherwise(sf.lower(sf.trim(sf.regexp_replace(sf.col("product_category_name"), r"_", r" ")))).alias("product_category_name"),

    sf.when(sf.col("product_name_lenght").isNull(),0)
             .otherwise(sf.abs(sf.col("product_name_lenght")))
            .alias("product_name_lenght"),
    
    sf.when(sf.col("product_description_lenght").isNull(),0)
            .otherwise(sf.abs(sf.col("product_description_lenght")))
            .alias("product_description_lenght"),

    sf.when(sf.col("product_weight_g").isNull(),0)
            .otherwise(sf.abs(sf.col("product_weight_g")))
            .alias("product_weight_g"),

    sf.when(sf.col("product_length_cm").isNull(),0)
            .otherwise(sf.abs(sf.col("product_length_cm")))
            .alias("product_length_cm"),

    sf.when(sf.col("product_photos_qty").isNull(),0)
            .otherwise(sf.abs(sf.col("product_photos_qty")))
            .alias("product_photos_qty"),

    sf.when(sf.col("product_width_cm").isNull(),0)
            .otherwise(sf.abs(sf.col("product_width_cm")))
            .alias("product_width_cm"),


    sf.when(sf.col("product_height_cm").isNull(),0)
            .otherwise(sf.abs(sf.col("product_height_cm")))
            .alias("product_height_cm"),
    )

In [0]:
df_products.write.mode("overwrite").parquet(products_write_path)

# PRODUCT CATEGORY NAME TRANSLATIONS TRANSFORMATION

In [0]:
df_product_category_name_translation= spark.read.csv(product_category_name_translation_read_path, header=True, inferSchema=True)

In [0]:
df_product_category_name_translation=df_product_category_name_translation.distinct()

df_product_category_name_translation = df_product_category_name_translation.dropDuplicates(["product_category_name"])

df_product_category_name_translation= df_product_category_name_translation.where(
    sf.col("product_category_name").isNotNull() & 
    (sf.trim(sf.col("product_category_name")) != "") &
    sf.col("product_category_name_english").isNotNull() &
    (sf.trim(sf.col("product_category_name_english")) != "")
)

df_product_category_name_translation=df_product_category_name_translation.select(
    sf.lower(sf.trim(sf.regexp_replace(sf.col("product_category_name"), r"_", r" "))).alias("product_category_name"),
    sf.lower(sf.trim(sf.regexp_replace(sf.col("product_category_name_english"), r"_", r" "))).alias("product_category_name_english")
)

In [0]:
df_product_category_name_translation.write.mode("overwrite").parquet(product_category_name_translation_write_path)

# ORDERS

# ORDER REVIEWS TRANSFORMATION

In [0]:
df_order_reviews= spark.read.csv(order_reviews_read_path, header=True, inferSchema=True)

In [0]:
df_order_reviews=df_order_reviews.distinct()

df_order_reviews = df_order_reviews.dropDuplicates(["review_id"]) 
# I removed duplicates because every review ID must be unique." or "Since every review ID must be unique, we eliminated the duplicates.
df_order_reviews = df_order_reviews.dropDuplicates(["order_id"]) 
# "Each order can only have one review. 
# Because it is an order assessment, not a product assessment, it must come from the person who placed the order. Even if the same products were ordered 
# simultaneously, the order ID for each would be unique/different

df_order_reviews= df_order_reviews.withColumn("review_score", sf.col("review_score").try_cast("int"))
df_order_reviews=df_order_reviews.withColumn("review_creation_date",sf.try_to_timestamp(sf.col("review_creation_date"),sf.lit("yyyy-MM-dd HH:mm:ss")))
df_order_reviews=df_order_reviews.withColumn("review_answer_timestamp",sf.try_to_timestamp(sf.col("review_answer_timestamp"),sf.lit("yyyy-MM-dd HH:mm:ss")))


df_order_reviews=df_order_reviews.where(
    (sf.col("review_id").isNotNull())&
    (sf.trim(sf.col("review_id"))!="")&
    ~(sf.col("review_id").contains(" "))& 
    (sf.col("order_id").isNotNull())&
    (sf.trim(sf.col("order_id"))!="")&
    ~(sf.col("order_id").contains(" "))& 
    (sf.col("review_score").isNotNull())&
    (sf.col("review_creation_date").isNotNull())  
)


df_order_reviews= df_order_reviews.select(
    sf.lower(sf.trim(sf.col("review_id"))).alias("review_id"),
    sf.lower(sf.trim(sf.col("order_id"))).alias("order_id"),
    sf.when(sf.col("review_score")<0,sf.abs(sf.col("review_score"))).otherwise(sf.col("review_score")).alias("review_score"),
    sf.when(sf.col("review_comment_title").isNull(),sf.lit("No Title")).otherwise(sf.lower(sf.trim(sf.col("review_comment_title")))).alias("review_comment_title"),
    sf.when(sf.col("review_comment_message").isNull(),sf.lit("No Message")).otherwise(sf.lower(sf.trim(sf.col("review_comment_message")))).alias("review_comment_message"),
    sf.col("review_creation_date"),
    sf.col("review_answer_timestamp")
)

In [0]:
df_order_reviews.write.mode("overwrite").parquet(order_reviews_write_path)

# PRODUCT ORDER ITEMS TRANSFORMATION

In [0]:
df_order_items= spark.read.csv(order_items_read_path, header=True, inferSchema=True)

In [0]:
df_order_items=df_order_items.distinct()

df_order_items = df_order_items.dropDuplicates(["order_id","order_item_id"])

df_order_items= df_order_items.where(
    (sf.col("order_id").isNotNull())&
    (sf.trim(sf.col("order_id"))!="")&
    ~(sf.col("order_id").contains(" "))& 

    (sf.col("order_item_id").isNotNull())&
    (sf.col("order_item_id")!=0)&
    (sf.col("order_item_id")>0)&

    (sf.col("product_id").isNotNull())&
    (sf.trim(sf.col("product_id"))!="")&
    ~(sf.col("product_id").contains(" "))& 

    (sf.col("seller_id").isNotNull())&
    (sf.trim(sf.col("seller_id"))!="")&
    ~(sf.col("seller_id").contains(" "))& 

    (sf.col("shipping_limit_date").isNotNull()) &

    (sf.col("price").isNotNull())&
    (sf.col("price")>0)&

    (sf.col("freight_value").isNotNull())&
    (sf.col("freight_value")>=0	)
)


df_order_items= df_order_items.select(
    sf.lower(sf.trim(sf.col("order_id"))).alias("order_id"),
    sf.col("order_item_id").alias("order_item_id"),
    sf.lower(sf.trim(sf.col("product_id"))).alias("product_id"),
    sf.lower(sf.trim(sf.col("seller_id"))).alias("seller_id"),
    sf.col("shipping_limit_date").alias("shipping_limit_date"),
    sf.col("price").alias("price"),
    sf.col("freight_value").alias("freight_value")
)

In [0]:
df_order_items.write.mode("overwrite").parquet(order_items_write_path)

# ORDER PAYMENTS TRANSFORMATION

In [0]:
df_order_payments= spark.read.csv(order_payments_read_path, header=True, inferSchema=True)

In [0]:
from pyspark.sql.window import Window

df_order_payments= df_order_payments.distinct()
df_order_payments= df_order_payments.dropDuplicates(["order_id","payment_sequential"])
df_order_payments= df_order_payments.withColumn("order_id_count", sf.count(sf.lit(1)).over(Window.partitionBy("order_id")))

df_order_payments= df_order_payments.where(
    (sf.col("order_id").isNotNull())&
    (sf.trim(sf.col("order_id"))!="")&
    ~(sf.col("order_id").contains(" "))
)


valid_payment_types = ["credit_card", "boleto", "voucher", "debit_card"]


df_order_payments= df_order_payments.select(
    sf.lower(sf.trim(sf.col("order_id"))).alias("order_id"),

    (sf.when(sf.col("payment_sequential").isNull(),sf.col("order_id_count")).otherwise(sf.abs(sf.col("payment_sequential")))).alias("payment_sequential"),

    (sf.when(
        sf.lower(sf.trim(sf.col("payment_type"))).isin(valid_payment_types),sf.lower(sf.trim(sf.col("payment_type")))
        ).otherwise("unknown").alias("payment_type")),

    sf.when(sf.col("payment_installments").isNull(),1).otherwise(sf.abs(sf.col("payment_installments"))).alias("payment_installments"),
    sf.when(sf.col("payment_value").isNull(),0).otherwise(sf.abs(sf.col("payment_value"))).alias("payment_value")
    
)

In [0]:
df_order_payments.write.mode("overwrite").parquet(order_payments_write_path)

# ORDERS TRANSFORMATION

In [0]:
df_orders= spark.read.csv(orders_read_path, header=True, inferSchema=True)

In [0]:
df_orders= df_orders.distinct()

df_orders=df_orders.dropDuplicates(["order_id"])

df_orders= df_orders.where(
    (sf.col("order_id").isNotNull())&
    (sf.trim(sf.col("order_id"))!="")&
   (~(sf.col("order_id").contains(" ")))&

   (sf.col("customer_id").isNotNull())&
    (sf.trim(sf.col("customer_id"))!="")&
   (~(sf.col("customer_id").contains(" ")))&

   (sf.col("order_purchase_timestamp").isNotNull())&

   (
       (sf.col("order_approved_at").isNull()) |
       (sf.col("order_purchase_timestamp")<sf.col("order_approved_at"))
   ) &

   (
       (sf.col("order_delivered_customer_date").isNull()) |
       (sf.col("order_delivered_carrier_date")<sf.col("order_delivered_customer_date"))
   )&

   (
       (sf.col("order_estimated_delivery_date").isNull()) |
       (sf.col("order_approved_at")<sf.col("order_estimated_delivery_date"))
   )&
   
   (
       sf.col("order_delivered_customer_date").isNull() |
       (sf.col("order_delivered_customer_date")<sf.now())
   )

   
)

valid_order_status=["created","delivered","canceled","invoiced","processing","shipped","approved"]

df_orders= df_orders.select(
    sf.lower(sf.trim(sf.col("order_id"))).alias("order_id"),
    sf.lower(sf.trim(sf.col("customer_id"))).alias("customer_id"),

    sf.when(sf.lower(sf.trim(sf.col("order_status"))).isin(valid_order_status),sf.lower(sf.trim(sf.col("order_status"))))
    .otherwise(sf.lit("unknown")).alias("order_status"),
    
    sf.col("order_purchase_timestamp"),
    sf.col("order_approved_at"),
    sf.col("order_delivered_carrier_date"),
    sf.col("order_delivered_customer_date"),
    sf.col("order_estimated_delivery_date")
)


In [0]:
df_orders.write.mode("overwrite").parquet(orders_write_path)